In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load the telemetry data
metric_df = pd.read_csv('metric.csv')
log_df = pd.read_csv('log.csv')
trace_df = pd.read_csv('trace.csv')
error_logs_df = pd.read_csv('error_logs.csv')

# Convert timestamp to datetime and set timezone to UTC+8
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s').dt.tz_localize('UTC').dt.tz_convert('Asia/Shanghai')
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s').dt.tz_localize('UTC').dt.tz_convert('Asia/Shanghai')
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s').dt.tz_localize('UTC').dt.tz_convert('Asia/Shanghai')
error_logs_df['timestamp'] = pd.to_datetime(error_logs_df['timestamp'], unit='s').dt.tz_localize('UTC').dt.tz_convert('Asia/Shanghai')

# Aggregate metrics by component and KPI
metric_agg = metric_df.groupby(['cmdb_id', 'kpi_name']).agg(
    count=('value', 'count'),
    min_value=('value', 'min'),
    max_value=('value', 'max'),
    p95=('value', lambda x: x.quantile(0.95)),
    earliest_timestamp=('timestamp', 'min')
).reset_index()

# Aggregate logs by component and log name
log_agg = log_df.groupby(['cmdb_id', 'log_name']).agg(
    count=('value', 'count'),
    min_value=('value', 'min'),
    max_value=('value', 'max'),
    p95=('value', lambda x: x.quantile(0.95)),
    earliest_timestamp=('timestamp', 'min')
).reset_index()

# Aggregate traces by component and trace name
trace_agg = trace_df.groupby(['cmdb_id', 'trace_name']).agg(
    count=('value', 'count'),
    min_value=('value', 'min'),
    max_value=('value', 'max'),
    p95=('value', lambda x: x.quantile(0.95)),
    earliest_timestamp=('timestamp', 'min')
).reset_index()

# Combine all aggregated data into a single DataFrame
combined_agg = pd.concat([metric_agg, log_agg, trace_agg], ignore_index=True)

# Display the top 20 aggregated results
combined_summary = combined_agg.head(20)
combined_summary

```
Out[1]:
```
The aggregation of telemetry data has been successfully completed, resulting in multiple time series classified by 'component-KPI'. The global thresholds (P95) for each 'component-KPI' have also been calculated. The summary includes various metrics such as count, minimum and maximum values, and the earliest timestamp for each KPI across different components, including 'carts', 'carts-db', 'catalogue', and 'catalogue-db'. The data is ready for further analysis to identify potential issues or anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count     min_value     max_value           p95        earliest_timestamp log_name trace_name
0          carts         cpu     25  1.170727e+00  7.958958e+01  7.934422e+01 2024-01-19 22:52:00+08:00      NaN        NaN
1          carts      diskio     25  0.000000e+00  1.163966e+05  5.924902e+04 2024-01-19 22:52:00+08:00      NaN        NaN
2          carts       error      5  0.000000e+00  0.000000e+00  0.000000e+00 2024-01-19 23:12:00+08:00      NaN        NaN
3          carts  latency-50     25  1.266706e-02  2.043347e-02  1.948694e-02 2024-01-19 22:52:00+08:00      NaN        NaN
4          carts  latency-90     25  2.273448e-02  6.528074e-02  4.712160e-02 2024-01-19 22:52:00+08:00      NaN        NaN
5          carts         mem     25  2.162033e+08  2.224593e+08  2.219550e+08 2024-01-19 22:52:00+08:00      NaN        NaN
6          carts      socket     25  9.500000e+00  1.343333e+01  1.305333e+01 2024-01-19 22:52:00+08:00      NaN        NaN
7          carts    workload     25  7.641183e+00  8.655417e+00  8.486297e+00 2024-01-19 22:52:00+08:00      NaN        NaN
8       carts-db         cpu     25  3.034738e+00  4.388721e+00  4.275059e+00 2024-01-19 22:52:00+08:00      NaN        NaN
9       carts-db      diskio     25  1.790057e+06  2.296373e+06  2.270569e+06 2024-01-19 22:52:00+08:00      NaN        NaN
10      carts-db         mem     25  7.884390e+07  8.425316e+07  8.396829e+07 2024-01-19 22:52:00+08:00      NaN        NaN
11      carts-db      socket     25  6.000000e+00  7.000000e+00  7.000000e+00 2024-01-19 22:52:00+08:00      NaN        NaN
12     catalogue         cpu     25  1.336217e-01  1.858281e-01  1.833097e-01 2024-01-19 22:52:00+08:00      NaN        NaN
13     catalogue  latency-50     25  3.002940e-03  3.110457e-03  3.095734e-03 2024-01-19 22:52:00+08:00      NaN        NaN
14     catalogue  latency-90     25  4.605291e-03  4.798823e-03  4.772321e-03 2024-01-19 22:52:00+08:00      NaN        NaN
15     catalogue         mem     25  6.176768e+06  6.272137e+06  6.267372e+06 2024-01-19 22:52:00+08:00      NaN        NaN
16     catalogue      socket     25  6.000000e+00  6.000000e+00  6.000000e+00 2024-01-19 22:52:00+08:00      NaN        NaN
17     catalogue    workload     25  3.925483e+00  4.323467e+00  4.260883e+00 2024-01-19 22:52:00+08:00      NaN        NaN
18  catalogue-db         cpu     25  1.749846e-01  1.977208e-01  1.954904e-01 2024-01-19 22:52:00+08:00      NaN        NaN
19  catalogue-db      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00 2024-01-19 22:52:00+08:00      NaN        NaN```
```

In [ ]:
# In[2]:

# Define the incident window in UTC+8 timezone
incident_start = pd.Timestamp('2024-01-19 14:49:49').tz_localize('Asia/Shanghai')
incident_end = pd.Timestamp('2024-01-19 15:19:49').tz_localize('Asia/Shanghai')

# Filter the combined aggregated data for the incident window
filtered_data = combined_agg[(combined_agg['earliest_timestamp'] >= incident_start) & 
                              (combined_agg['earliest_timestamp'] <= incident_end)]

# Prepare the filtered data for anomaly detection
anomaly_prepared_data = filtered_data[['cmdb_id', 'kpi_name', 'count', 'min_value', 'max_value', 'p95', 'earliest_timestamp']]

# Display the top 20 rows of the prepared data for anomaly detection
anomaly_prepared_summary = anomaly_prepared_data.head(20)
anomaly_prepared_summary

```
Out[2]:
```
The filtering of the aggregated telemetry data for the specified incident window has resulted in an empty dataset. This indicates that there are no recorded time series within the incident window of 2024-01-19 14:49:49 to 2024-01-19 15:19:49 UTC. Therefore, there are no data points available for anomaly detection during this period.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, kpi_name, count, min_value, max_value, p95, earliest_timestamp]
Index: []```
```

In [ ]:
# In[3]:

# Load the error logs and log data
error_logs_df = pd.read_csv('error_logs.csv')
log_df = pd.read_csv('log.csv')

# Convert timestamp to datetime and set timezone to UTC+8
error_logs_df['timestamp'] = pd.to_datetime(error_logs_df['timestamp'], unit='s').dt.tz_localize('UTC').dt.tz_convert('Asia/Shanghai')
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s').dt.tz_localize('UTC').dt.tz_convert('Asia/Shanghai')

# Define the incident window in UTC+8 timezone
incident_start = pd.Timestamp('2024-01-19 14:49:49').tz_localize('Asia/Shanghai')
incident_end = pd.Timestamp('2024-01-19 15:19:49').tz_localize('Asia/Shanghai')

# Filter error logs and log data for the incident window
filtered_error_logs = error_logs_df[(error_logs_df['timestamp'] >= incident_start) & 
                                     (error_logs_df['timestamp'] <= incident_end)]

filtered_log = log_df[(log_df['timestamp'] >= incident_start) & 
                      (log_df['timestamp'] <= incident_end)]

# Summarize findings
error_log_summary = filtered_error_logs.groupby('cmdb_id').agg(
    count=('message', 'count'),
    earliest_timestamp=('timestamp', 'min')
).reset_index()

log_summary = filtered_log.groupby('cmdb_id').agg(
    count=('value', 'count'),
    earliest_timestamp=('timestamp', 'min')
).reset_index()

# Combine summaries
combined_summary = pd.concat([error_log_summary, log_summary], ignore_index=True)

# Display the top 20 rows of the combined summary
combined_summary_result = combined_summary.head(20)
combined_summary_result

```
Out[3]:
```
The scan of the `error_logs.csv` and `log.csv` files for entries within the specified incident window has resulted in an empty dataset. This indicates that there are no recorded error logs or log entries during the period of 2024-01-19 14:49:49 to 2024-01-19 15:19:49 UTC. Therefore, no relevant messages or counts are available from these logs for this timeframe.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, count, earliest_timestamp]
Index: []```
```

In [ ]:
# In[4]:

# Load the trace data
trace_df = pd.read_csv('trace.csv')

# Convert timestamp to datetime and set timezone to UTC+8
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s').dt.tz_localize('UTC').dt.tz_convert('Asia/Shanghai')

# Filter trace data for the incident window
filtered_trace = trace_df[(trace_df['timestamp'] >= incident_start) & 
                          (trace_df['timestamp'] <= incident_end)]

# Summarize findings
trace_summary = filtered_trace.groupby('cmdb_id').agg(
    count=('value', 'count'),
    min_value=('value', 'min'),
    max_value=('value', 'max'),
    earliest_timestamp=('timestamp', 'min')
).reset_index()

# Display the top 20 rows of the trace summary
trace_summary_result = trace_summary.head(20)
trace_summary_result

```
Out[4]:
```
The scan of the `trace.csv` file for entries within the specified incident window has resulted in an empty dataset. This indicates that there are no recorded trace entries during the period of 2024-01-19 14:49:49 to 2024-01-19 15:19:49 UTC. Therefore, no relevant trace details or counts are available for this timeframe.

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [cmdb_id, count, min_value, max_value, earliest_timestamp]
Index: []```
```